# Train your own CodeDreamer in ~10 minutes

[**CodeDreamer**](https://github.com/) is a *grounded latent world model of code execution*
(Python package `execwm`): a small neural net that runs programs **in its head**, predicting how
machine state evolves in a learned latent space while every step is anchored to the exact output of
a real interpreter.

In this notebook we train a **tiny** CodeDreamer from scratch on CPU and reproduce the core finding
of the project: when you push programs to **out-of-distribution magnitudes**, whole-state exact-match
collapses — but if you offload the arithmetic to a perfect ALU and let the net keep predicting
*structure* (which slot changes, the op type, the sign, the next pc, branch outcomes), accuracy holds
up. In other words, **the magnitude wall is the learned digit head, not the latent**. We see the same
effect here even at a fraction of the full training budget.

In [ ]:
# --- Setup ---
# On Google Colab, first clone the repo and cd into it:
#   !git clone https://github.com/<org>/CodeDreamer.git
#   %cd CodeDreamer
# (here the repo is already checked out, so we skip the clone)

!pip install -q numpy torch

import sys
# This notebook lives in notebooks/, so the repo root is one level up.
sys.path.insert(0, '..')

import torch
print('torch', torch.__version__)

## 1. Build the spec + codec and train a tiny model

We use exactly the setup from `scripts/neurosym_spike.py`:

- a **small-magnitude** `GenSpec` (only `ADD`/`SUB`, `max_const = max_input_val = 5`, so register
  values stay below ~30), and
- a **wide** `CodecConfig` (`max_digits=4`, i.e. the codec can represent numbers up to 9999).

The codec is deliberately wider than the training data needs — that is what lets us later probe
*magnitude generalization* on the same frozen weights. We train with deliberately **small** settings
(300 steps, 800 train / 200 eval episodes, `d_model=128`, 2 encoder + 2 dynamics layers) so the whole
notebook runs on CPU in well under ~10 minutes.

In [ ]:
from execwm.data.state_codec import CodecConfig
from execwm.substrate.generators import GenSpec
from execwm.substrate.vm import Op
from execwm.train.train_m1 import TrainConfig, train


def small_spec(max_const: int, max_input_val: int) -> GenSpec:
    # Exactly the spec used in scripts/neurosym_spike.py: ADD/SUB only, heap on.
    return GenSpec(num_vars=4, num_inputs=2, num_temps=10, max_depth=2, num_stmts=5,
                   max_const=max_const, max_input_val=max_input_val, max_loop_count=3,
                   arith_ops=(Op.ADD, Op.SUB), use_heap=True, num_lists=1, list_len=4,
                   max_steps=128)


# In-distribution: small magnitudes (values <= ~30) ...
train_spec = small_spec(max_const=5, max_input_val=5)
# ... but a WIDE codec that can represent up to 9999 (max_digits=4).
codec = CodecConfig(max_digits=4, base=10, max_pc=128)

# Tiny budget so this runs on CPU in well under ~10 minutes.
tc = TrainConfig(steps=300, batch_size=48, max_len=18, lr=4e-4,
                 rollout_warmup=60, rollout_grow_every=120, rollout_max_k=6)

device = torch.device('cpu')
out = train(spec=train_spec, codec_cfg=codec, tc=tc,
            n_train=800, n_eval=200, seed=0, device=device, log_every=50,
            d_model=128, n_heads=4, enc_layers=2, dyn_layers=2)

model, scodec, acodec = out['model'], out['scodec'], out['acodec']
print('\nIn-distribution training eval:', out['eval'])

## 2. The neurosymbolic readout: in-distribution vs magnitude-OOD

Now we read the **same frozen model** two ways on two eval splits:

- **in-distribution** — the small spec it trained on (values <= ~30), and
- **magnitude-OOD** — identical *structure*, but `max_const = max_input_val = 400`, so values land
  around 300-800 (10-25x larger than anything seen in training).

`field_breakdown` reports, over valid transitions:

- `em_learned` — whole-state exact-match with **every** field decoded by the net (the status-quo readout);
- `em_digits_oracle` — the net's predictions, but the numeric **digit** payload is supplied by a perfect
  ALU (arithmetic offloaded). pc / type / sign / flags / which-slot-changed are **still** the net's job;
- `pc` — next-pc accuracy (pure control flow); and
- `written_digits` — digit accuracy of the written register (the arithmetic that fails OOD).

In [ ]:
from execwm.data.dataset import collect_examples
from execwm.eval.neurosym import field_breakdown

# In-distribution eval set (same small spec as training).
indist_ex, _ = collect_examples(train_spec, 200, lambda ex: True, 99, scodec, acodec)

# Magnitude-OOD eval set: same structure, but values ~300-800.
ood_spec = small_spec(max_const=400, max_input_val=400)
ood_ex, ood_att = collect_examples(ood_spec, 200, lambda ex: True, 777, scodec, acodec)
print(f'in-dist eval episodes {len(indist_ex)}; '
      f'OOD eval episodes {len(ood_ex)} (from {ood_att} attempts)')

indist = field_breakdown(model, indist_ex, scodec, acodec, device, max_len=18)
ood = field_breakdown(model, ood_ex, scodec, acodec, device, max_len=18)


def fmt(d, k):
    return f"{d.get(k, float('nan')):.3f}"


print('\n| split | EM_learned | EM_digits_oracle | pc | written_digits | n |')
print('|---|---|---|---|---|---|')
for name, d in [('in-distribution (val<=30)', indist),
                ('magnitude-OOD (val~300-800)', ood)]:
    print(f"| {name} | {fmt(d,'em_learned')} | {fmt(d,'em_digits_oracle')} | "
          f"{fmt(d,'pc')} | {fmt(d,'written_digits')} | {d.get('n',0)} |")

## 3. Interpretation: the wall is the digit head

Read the table above:

- **`em_learned` drops sharply OOD.** Forced to decode the exact digits of values it never saw, the
  net's whole-state exact-match collapses toward 0. This is the well-known magnitude wall.
- **`em_digits_oracle` and `pc` hold up much better OOD.** Once arithmetic is offloaded to the ALU,
  the net's *structural* prediction — which slot is written, the op type/sign, the next pc, branch
  outcomes — is largely magnitude-invariant. `written_digits` is the field that fails; the rest survives.

Even at this tiny budget (300 steps, `d_model=128`), the gap between `em_learned` and
`em_digits_oracle` on the OOD split is the whole point: **the magnitude wall lives in the learned
digit decoder, not in the latent.** The latent encodes the structure of the transition correctly
across magnitude; only the readout of numbers fails — so arithmetic should be offloaded to the
interpreter's ALU.

The full-scale result (d=256, ~10M params, 1500 steps: EM 0.000 -> 0.790 OOD on frozen weights) is
written up in [`FINDINGS_NEUROSYM.md`](../FINDINGS_NEUROSYM.md). Run the headline experiment with
`PYTHONPATH=. python scripts/neurosym_spike.py`, and try the interactive demo with
`PYTHONPATH=. python demo/app.py`.